In [ ]:
import re
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import CountVectorizer

import numpy as np
import pandas as pd

import nltk
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords as nltk_stopwords
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA, TruncatedSVD as SVD

In [ ]:
tfidf = pd.read_csv('tfidf.csv')
tfidfL2 = pd.read_csv('tfidfL2.csv')
lib = pd.read_csv('LIB.csv')
corpus = pd.read_csv('CORPUS.csv')
vocab = pd.read_csv('VOCAB.csv')

In [3]:
pca = PCA(n_components=8)

In [4]:
dcm = pd.DataFrame(pca.fit_transform(tfidf), index=tfidf.index)
dcm.columns = ['PC{}'.format(i) for i in dcm.columns]

In [5]:
comps = pd.DataFrame(pca.components_.T * np.sqrt(pca.explained_variance_))
comps.columns = ["PC{}".format(i) for i in comps.columns]
comps.index = tfidfL2.columns
comps.index.name = 'term_str'
comps.T

term_str,album_id,song_id,10,1159,12,123,17,23,3,300,...,za,zbog,zedd,zip,zippy,zlo,zombi,zombie,zombieboy,zone
PC0,-2.687699,37.671673,-0.021290,-0.041645,-0.013544,-0.023655,-0.021840,-0.021582,-0.022168,-0.014370,...,-0.024339,-0.027413,-0.021035,-0.020803,-0.023671,-0.024339,-0.024339,-0.061808,-0.049646,-0.022228
PC1,0.066019,0.104723,0.071564,0.070601,0.057577,0.079240,0.070787,0.075202,0.074174,0.061072,...,0.070945,0.070103,0.067834,0.071796,0.070589,0.070945,0.070945,0.072377,0.072200,0.058754
PC2,-0.066155,-0.003612,-0.001178,-0.003019,0.024727,-0.001662,-0.001373,-0.001639,0.032022,-0.001042,...,-0.001614,-0.002135,-0.001221,-0.001396,-0.001257,-0.001614,-0.001614,-0.005277,-0.004022,-0.001318
PC3,0.069704,-0.002524,-0.002504,-0.000413,-0.001519,-0.005173,-0.001211,-0.005010,-0.000557,-0.002784,...,-0.001267,-0.000316,-0.001287,-0.003415,-0.001676,-0.001267,-0.001267,0.001497,0.000383,-0.000430
PC4,0.012007,0.000508,0.001337,0.004653,0.000322,0.003659,0.001318,0.001350,0.000872,0.000863,...,0.000125,-0.001111,0.000923,-0.001415,0.001148,0.000125,0.000125,0.009211,0.006856,0.001148
PC5,-0.052483,-0.004890,-0.001658,-0.003645,-0.001691,-0.002986,-0.001825,-0.003378,-0.001036,-0.002051,...,-0.001330,-0.001159,-0.001191,-0.000338,-0.001409,-0.001330,-0.001330,-0.005377,-0.004215,-0.001314
PC6,0.050778,0.005290,-0.001121,-0.003240,0.000737,-0.000601,-0.001300,-0.002796,-0.000397,0.000853,...,0.018889,0.038525,-0.001638,-0.004335,-0.001243,0.018889,0.018889,-0.003457,-0.002644,-0.001938
PC7,0.063209,0.003300,0.000315,0.004305,0.000850,0.000864,0.000664,0.001299,0.000631,0.001237,...,-0.007760,-0.016120,0.000767,-0.000407,0.000738,-0.007760,-0.007760,0.011394,0.008156,0.000763


In [6]:
import matplotlib.pyplot as plt
import plotly_express as px

def vis_loadings(comps, a=0, b=1, hover_name='term_str'):
    return px.scatter(comps.reset_index(), f"PC{a}", f"PC{b}", 
                      text='term_str', 
                      marginal_x='box', 
                      height=800)

In [7]:
def vis_pcs(dcm, a, b, label='album', hover_name='track_name'):
    return px.scatter(dcm, f"PC{a}", f"PC{b}", 
                    color=lib[label], 
                    hover_name=lib[hover_name], 
                    size=lib['n_tokens'],
                    marginal_x='box', height=800)

In [ ]:
keep = (vocab[~vocab["stop"]]
        .sort_values("dfidf", ascending=False)
        .head(len(vocab))["term_str"]
        .tolist())

X = tfidf

X_centered = X - X.mean(axis=0)

loadings = pd.DataFrame(
    pca.components_.T,
    index=pd.Index(X.columns, name="term_str"),
    columns=[f"PC{i}" for i in range(8)],
)

vocab_lookup = vocab.set_index("term_str")[["n", "df", "dfidf", "max_pos_group"]]
loadings = loadings.join(vocab_lookup, how="left")

var = pd.DataFrame({
    "pc": [f"PC{i}" for i in range(8)],
    "eigenvalue": pca.explained_variance_,
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative": np.cumsum(pca.explained_variance_ratio_),
}).set_index("pc")

In [ ]:
scores = pca.transform(X_centered.values)
SONG_PCA = pd.DataFrame(
    scores,
    index=X.index,
    columns=[f"PC{i}" for i in range(8)],
)
SONG_PCA = SONG_PCA.join(lib[["track_name", "album"]], on="song_id")

### PCA VIS 1

In [ ]:
pc1_var = var.loc["PC0", "explained_variance_ratio"]
pc2_var = var.loc["PC1", "explained_variance_ratio"]

fig, ax = plt.subplots(figsize=(10, 7))

# Order albums chronologically so the legend reads left-to-right in time
album_order = (lib.sort_values("release_date")["album"]
               .drop_duplicates()
               .tolist())

for album in album_order:
    sub = SONG_PCA[SONG_PCA["album"] == album]
    if sub.empty:
        continue
    ax.scatter(sub["PC0"], sub["PC1"], s=45, alpha=0.8, label=album)

x_center, y_center = -175, 1
x_span, y_span = 20, 1

ax.set_xlim(x_center - x_span, x_center + x_span)
ax.set_ylim(y_center - y_span, y_center + y_span)
ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="k", lw=0.5)
ax.set_xlabel(f"PC0 ({pc1_var:.1%} variance)")
ax.set_ylabel(f"PC1 ({pc2_var:.1%} variance)")
ax.set_title("Gaga songs in PC0 PC1 space, colored by album")
ax.legend(fontsize=8, loc="best", ncol=2, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
def vis_loadings(comps, a=0, b=1, hover_name="term_str"):
    return px.scatter(
        comps.reset_index(),
        x=f"PC{a}",
        y=f"PC{b}",
        text="term_str",
        hover_name=hover_name,
        marginal_x="box",
        height=800,
    )

fig = vis_loadings(loadings, a=0, b=1)
fig.show()

### PCA VIS 2

In [ ]:
pc1_var = var.loc["PC2", "explained_variance_ratio"]
pc2_var = var.loc["PC3", "explained_variance_ratio"]

fig, ax = plt.subplots(figsize=(10, 7))

album_order = (lib.sort_values("release_date")["album"]
               .drop_duplicates()
               .tolist())

for album in album_order:
    sub = SONG_PCA[SONG_PCA["album"] == album]
    if sub.empty:
        continue
    ax.scatter(sub["PC2"], sub["PC3"], s=45, alpha=0.8, label=album)

ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="k", lw=0.5)
ax.set_xlabel(f"PC2 ({pc1_var:.1%} variance)")
ax.set_ylabel(f"PC3 ({pc2_var:.1%} variance)")
ax.set_title("Gaga songs in PC2 - PC3 space, colored by album")
ax.legend(fontsize=8, loc="lower left", ncol=2, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
fig = vis_loadings(loadings, a=2, b=3)
fig.show()

### LDA

In [ ]:
docs = (
    corpus.reset_index()
          .groupby(['album_id', 'song_id'])['term_str']
          .apply(' '.join)
          .to_frame('term_str')
)
docs.head()

In [ ]:
from sklearn.feature_extraction import text
from sklearn.decomposition import LatentDirichletAllocation as LDA, NMF

my_stop_words = list(text.ENGLISH_STOP_WORDS.union(['yes']))

count_engine = CountVectorizer(max_df=.9, min_df=10, stop_words=my_stop_words)
count_model = count_engine.fit_transform(docs.term_str)

n_topics = 8
max_iter = 100
n_top_terms = 10

TNAMES = [f"T{str(x).zfill(len(str(n_topics)))}" for x in range(n_topics)]
topic_engine = LDA(n_components=n_topics, max_iter=max_iter)
topic_model = topic_engine.fit_transform(count_model)


### THETA

In [ ]:
THETA = pd.DataFrame(topic_model, index=docs.index, columns=TNAMES)
THETA.columns.name = 'topic_id'
THETA.sample(10).T.style.background_gradient(cmap="YlGnBu", axis=None)

### PHI

In [ ]:
TERMS = count_engine.get_feature_names_out()
PHI = pd.DataFrame(topic_engine.components_, columns=TERMS, index=TNAMES)
PHI.index.name = 'topic_id'
PHI.columns.name = 'term_str'
PHI.T.sample(10).T.style.background_gradient(cmap='YlGnBu', axis=None)

### LDA x PCA Vis

In [ ]:
TOPICS = PHI.stack().groupby('topic_id')\
    .apply(lambda x: ' '.join(x.sort_values(ascending=False).head(n_top_terms).reset_index().term_str))\
    .to_frame('top_terms')
TOPICS

In [ ]:
X = THETA.T
pca = PCA(n_components=2)
topic_coords = pca.fit_transform(X)

TOPIC_PCA = pd.DataFrame(
    topic_coords,
    index=THETA.columns,
    columns=['PC1', 'PC2'],
)
TOPIC_PCA['mean_weight'] = THETA.mean(axis=0)

META = 'album'

if lib.index.name == 'song_id':
    lib_lookup = lib[~lib.index.duplicated(keep='first')][META].astype(object)
else:
    lib_lookup = (lib.drop_duplicates(subset='song_id')
                     .set_index('song_id')[META].astype(object))

theta_docs = THETA
if isinstance(theta_docs.index, pd.MultiIndex) and 'song_id' in theta_docs.index.names:
    theta_docs = theta_docs.reset_index().set_index('song_id')[THETA.columns]

top_doc = theta_docs.idxmax(axis=0)
TOPIC_PCA['color'] = top_doc.map(lib_lookup)
cmap_kind = 'categorical'

print('topics:', TOPIC_PCA.shape[0])
print('unique album colors:', TOPIC_PCA['color'].nunique())
print('NaN colors:', TOPIC_PCA["color"].isna().sum())
TOPIC_PCA.head()

In [ ]:

fig, ax = plt.subplots(figsize=(10, 7))
s = TOPIC_PCA["mean_weight"]
sizes = 40 + 2000 * (s - s.min()) / (s.max() - s.min() + 1e-12)

if cmap_kind == "numeric":
    sc = ax.scatter(
        TOPIC_PCA["PC1"], TOPIC_PCA["PC2"],
        s=sizes, c=TOPIC_PCA["color"],
        cmap="viridis", alpha=0.8, edgecolor="k", linewidth=0.5,
    )
    plt.colorbar(sc, ax=ax, label=META)
else:
    cats = TOPIC_PCA["color"].astype("category")
    sc = ax.scatter(
        TOPIC_PCA["PC1"], TOPIC_PCA["PC2"],
        s=sizes, c=cats.cat.codes,
        cmap="tab20", alpha=0.8, edgecolor="k", linewidth=0.5,
    )
    handles = [plt.Line2D([0], [0], marker='o', color='w',
                          markerfacecolor=plt.cm.tab20(i / max(len(cats.cat.categories)-1, 1)),
                          markersize=8, label=c)
               for i, c in enumerate(cats.cat.categories)]
    ax.legend(handles=handles, title=META,
              bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)

for tid, row in TOPIC_PCA.iterrows():
    ax.annotate(str(tid), (row["PC1"], row["PC2"]),
                fontsize=8, ha="center", va="center")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("Topics in PC1Ãƒâ€”PC2 space\n(size = mean doc weight, color = "
             f"{META})")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()